# Định Giá Đúng (The Price is Right)

## Lộ trình Tuần 8

Ngày 1: Modal.com và SpecialistAgent  
Ngày 2: RAG, FrontierAgent, Ensemble Agent  
Ngày 3: ScannerAgent, MessengerAgent  
Ngày 4: AutonomousPlannerAgent và DealAgentFramework  
Ngày 5: Chung kết The Price Is Right


Hôm nay chúng ta sẽ xây dựng thêm một mảnh ghép nữa: một ScanningAgent (agent quét dữ liệu) để tìm các deal hấp dẫn bằng cách theo dõi (subscribe) các nguồn RSS feed.

## 📝 Ghi chú tổng quan notebook

### Tóm tắt quy trình của notebook
Notebook này xây dựng và thử nghiệm 2 agent mới: (1) ScannerAgent - quét các nguồn RSS để lấy danh sách deal, dùng GPT-5-mini để chọn ra 5 deal có mô tả chi tiết và giá rõ ràng nhất; (2) MessagingAgent - gửi thông báo đẩy (push notification) qua Pushover khi tìm thấy deal tốt. Luồng xử lý: lấy dữ liệu thô từ RSS → xây prompt yêu cầu LLM lọc & tóm tắt deal → nhận kết quả có cấu trúc (JSON) → thử gửi thông báo qua Pushover.

### Ý nghĩa chính của notebook
Notebook giải quyết bài toán tự động hoá việc phát hiện deal tốt: dữ liệu thô (RSS feed) được thu thập, sau đó được một LLM lọc và chuẩn hoá thành các deal có mô tả rõ ràng kèm giá, cuối cùng được gói lại thành `ScannerAgent` để tái sử dụng. Song song đó, `MessagingAgent` đảm nhiệm việc thông báo cho người dùng qua điện thoại khi có deal đáng chú ý.

### Mục tiêu cuối cùng
Sau khi chạy xong notebook, ta có được `ScannerAgent` (quét & chọn lọc deal bằng LLM) và `MessagingAgent` (gửi thông báo đẩy) - hai thành phần sẽ được `AutonomousPlannerAgent` sử dụng ở Ngày 4 để tự động hoá toàn bộ quy trình săn deal.

In [ ]:
# Import thư viện cần thiết: os/dotenv để đọc biến môi trường, OpenAI để gọi LLM,
# ScrapedDeal/DealSelection là các model dữ liệu deal, logging để ghi log,
# requests để gửi HTTP request (dùng cho Pushover ở phần sau).
# Khởi tạo client OpenAI và chọn model gpt-5-mini (nhanh, rẻ, phù hợp để lọc dữ liệu).

import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [ ]:
# Quét (scrape) các nguồn RSS feed để lấy danh sách deal thô, có hiển thị thanh tiến trình.
deals = ScrapedDeal.fetch(show_progress=True)

In [ ]:
# Kiểm tra xem đã quét được bao nhiêu deal.
len(deals)

In [ ]:
# Xem thử mô tả chi tiết của một deal (deal số 11) để hiểu định dạng dữ liệu.
deals[10].describe()

### Chúng ta sẽ nhờ GPT-5-mini tóm tắt các deal và xác định giá của chúng

In [ ]:
# Định nghĩa các prompt (giữ nguyên tiếng Anh vì đây là nội dung gửi trực tiếp cho LLM):
# - SYSTEM_PROMPT: hướng dẫn LLM chọn ra 5 deal có mô tả chi tiết và giá rõ ràng nhất.
# - USER_PROMPT_PREFIX/SUFFIX: phần đầu và cuối của prompt người dùng, sẽ được ghép
#   với nội dung các deal đã quét được ở cell tiếp theo.

SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [ ]:
# Hàm này tạo prompt người dùng hoàn chỉnh từ danh sách các deal đã quét được:
# ghép phần mở đầu (PREFIX) + mô tả từng deal + phần kết (SUFFIX).

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [ ]:
# Tạo prompt người dùng từ các deal vừa quét được, và xem thử 2000 ký tự đầu tiên
# để kiểm tra định dạng trước khi gửi cho LLM. Sau đó ghép thành messages hoàn chỉnh.

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

In [ ]:
# Gọi GPT-5-mini với response_format=DealSelection để LLM trả về kết quả có cấu trúc
# (structured output), tự động parse thành object DealSelection thay vì text thô.
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

In [ ]:
# In ra danh sách 5 deal mà LLM đã chọn: mô tả sản phẩm, giá và URL của từng deal.
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


In [ ]:
# Bật ghi log ở mức INFO để theo dõi hoạt động bên trong ScannerAgent ở cell tiếp theo.
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
# Import class ScannerAgent - phiên bản đóng gói của toàn bộ pipeline quét & lọc deal ở trên.
from agents.scanner_agent import ScannerAgent

In [ ]:
# Khởi tạo ScannerAgent và gọi scan() để thực hiện toàn bộ quy trình:
# quét RSS, lọc bằng LLM, và trả về danh sách deal tốt nhất.
agent = ScannerAgent()
result = agent.scan()

In [ ]:
# Xem kết quả các deal mà ScannerAgent đã chọn ra.
result

### Giới thiệu Pushover

Pushover là một công cụ tiện lợi để gửi thông báo đẩy (Push Notification) tới điện thoại của bạn.

Việc thiết lập và cài đặt rất đơn giản!

Chỉ cần truy cập https://pushover.net/ và bấm 'Login or Signup' ở góc trên bên phải để đăng ký một tài khoản miễn phí, và tạo API key của bạn.

Sau khi đăng ký xong, ở màn hình chính, bấm "Create an Application/API Token", đặt tên bất kỳ (ví dụ AIEngineer) rồi bấm Create Application.

Sau đó thêm 2 dòng sau vào file `.env`:

PUSHOVER_USER=_điền key nằm ở góc trên bên phải trang chủ Pushover của bạn, thường bắt đầu bằng chữ u_  
PUSHOVER_TOKEN=_điền key khi bạn vào ứng dụng vừa tạo tên là Agents (hoặc tên bạn đặt), thường bắt đầu bằng chữ a_

Nhớ lưu file `.env`, và chạy lại `load_dotenv(override=True)` sau khi lưu để nạp lại các biến môi trường.

Cuối cùng, bấm "Add Phone, Tablet or Desktop" để cài đặt lên điện thoại của bạn.

In [ ]:
# Nạp lại biến môi trường sau khi đã thêm PUSHOVER_USER và PUSHOVER_TOKEN vào file .env.
load_dotenv(override=True)

In [ ]:
# Đọc key và token của Pushover từ biến môi trường, và khai báo URL API để gửi thông báo.
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
# Kiểm tra xem đã đọc được PUSHOVER_USER và PUSHOVER_TOKEN từ .env chưa,
# chỉ in ký tự đầu tiên để xác nhận mà không lộ toàn bộ key nhạy cảm.
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
# Hàm gửi thông báo đẩy: đóng gói user, token và nội dung message thành payload,
# rồi gửi HTTP POST request tới API của Pushover.
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
# Thử gửi một thông báo đẩy để kiểm tra hàm push() hoạt động đúng.
push("MASSIVE DEAL!!")

In [ ]:
# Import MessagingAgent - phiên bản đóng gói của cơ chế gửi thông báo Pushover ở trên,
# giúp dễ tái sử dụng ở các agent khác trong hệ thống.
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [ ]:
# Thử phương thức notify(): gửi thông báo có định dạng đẩy đủ hơn, gồm mô tả deal,
# giá bán, giá trị ước tính và URL - đây là định dạng sẽ dùng khi AutonomousPlannerAgent
# tìm thấy một deal đáng chú ý.
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")